# Liu & Dori showcase -- what each raster-to-vector number means

Tiny synthetic GT / prediction cases, run through the **real** scorer (`liu_dori.evaluate` +
`vector_metrics.evaluate_vector_metrics` on `vector_render`'s renders -- exactly what the benchmark
does), one picture and one value table per case. Companion to `docs/RasterToVectorEvaluation.md`
(section numbers below refer to it).

**Units.** Page space in pt at `dpi = 72`, so 1 pt = 1 raster px and the width floor is 2 pt. The
canonical GT line is the paper's Table 2 line, (10, 20)-(90, 20), width 8.

**Reading the pictures** (same conventions as the viewer's `vector_eval` layers):

- GT bars: translucent bands at their matching width W_eff (dotted outline = dashed GT); predicted
  bars: thin lines. **Both coloured by their own line's Q_v**: red 0 -> yellow 0.5 -> green 1.
- Purple dots = GT endpoints, cyan dots = predicted endpoints.
- Black segment = an accepted overlap c (it lies on the prediction); dotted legs = d1 / d2 (distance
  of c's ends to g), with l(c), d_overlap and Q_v(c) in the label.
- Dashed grey rectangle = a text label box (text / non-text split).

Every image is also saved under `outputs/liu_dori_showcase/<timestamp>/`.

## 0 - Bootstrap

In [ ]:
import math
import os, sys

_root = os.path.abspath(os.path.join('../..'))
if _root not in sys.path:
    sys.path.insert(0, _root)

from IPython.display import Image, Markdown, display

from rastervec.notebooks import _liu_dori_showcase_helpers as h

book = h.Book()
print('output:', book.root)


def show(path, width=1000):
    display(Image(filename=str(path), width=width))


def table(cases, keys):
    display(Markdown(h.summary_table(cases, keys)))

## 1 - Bars and the width floor (§2)

Both sides are reduced to straight **bars**: an `"l"` item is one bar; a curve is sampled every
0.5 raster px and RDP-simplified with eps = W_eff / 4, each piece its own bar (thicker line -> larger
eps -> fewer, longer bars). Nothing is chained or de-duplicated.

Every tolerance is relative to the GT width, and CAD hairlines have W = 0, so both sides use
**W_eff = max(W, 2 px x 72 / dpi)**. The last three rows are the same 0.6 pt offset prediction of a
W = 0 hairline scored at different raster dpi: at 72 dpi W_eff = 2 pt and it matches (Q_od < 1); at
150 / 300 dpi W_eff / 2 < 0.6 pt and it is a miss plus a false alarm. *The floor is what lets a
hairline be matched at all, and it is tied to the resolution P2 actually saw.*

In [ ]:
bars = h.run_all(h.bars_cases())
show(book.cases('1_bars', bars, ['GT bars', 'pred bars', 'W_eff(g)', 'overlaps', 'D_v', 'F_v', 'Q_od']))

## 2 - Matching (§3)

A GT bar g and a predicted bar k **overlap** when

1. their undirected angle is <= 15 deg (the angle gate -- our addition, not in the paper),
2. at least 2 of the 4 endpoints lie inside the other bar's area (distance <= its W_eff / 2),
3. the overlap segment c (on k) is non-empty, and its Hausdorff distance to g, d_overlap, is < W(g)/2.

The three endpoint cases decide where c is: all of k (k inside g), g's ends projected onto k
(g inside k), or from k's inside end to g's inside end (partial). The overlap only covers what the
two lines share -- the uncovered GT length (k inside g) and the overrunning prediction length
(g inside k / partial) are what lower Q_b(g) and Q_b(k) later.

The last three rows: a 20 deg piece whose ends both sit inside g is still rejected by the gate --
without it (`paper_mode`) short pieces at corners and crossings would match perpendicular lines.

In [ ]:
matching = h.run_all(h.matching_cases())
show(book.cases('2_matching', matching, ['overlaps', 'Q_b(g)', 'Q_b(k)', 'Q_od', 'D_v', 'F_v', '|D(g)|', '|G(k)|']))

## 3 - Per-overlap quality factors (§4)

Per overlap c, all relative to W(g):

| factor | formula | reacts to |
|---|---|---|
| Q_pt | exp(-(d1 + d2) / W) | perpendicular distance of **c's two ends** to g -- an offset or tilted prediction |
| Q_od | exp(-2 d_overlap / W) | the **worst** deviation along c -- offset, skew, wobble |
| Q_w | exp(-\|W(k) - W(g)\| / W(g)) | width disagreement only |
| Q_v(c) | (Q_pt Q_od Q_w)^(1/3) | their geometric mean |

What the table makes visible:

- **Q_pt is not an extent measure.** c only spans the part both lines share, so its ends lie *on* g
  for a short or overshooting prediction: Q_pt = 1 for both (rows "short" and "overshoot"). Extent
  errors show up in Q_b(g) (too short) and Q_b(k) / F_v (too long), never in Q_pt.
- Since d_overlap >= max(d1, d2), **Q_pt >= Q_od always**: a parallel offset gives Q_pt = Q_od; a
  skew (one end off) gives Q_pt > Q_od.
- The headline Q_* rows are averages over **overlaps that exist** (weighted by l(c)): misses never
  enter. "short" has every factor = 1 and still D_v = 0.5.

In [ ]:
factors = h.run_all(h.factor_cases())
show(book.cases('3_factors', factors, ['Q_pt', 'Q_od', 'Q_w', 'Q_v(c)', 'Q_b(g)', 'Q_b(k)', 'D_v', 'F_v']))

Sweeps: the measured factor (markers) against its closed form (dashed). W = 8, so the offset limit is W/2 = 4 -- past it the overlap disappears and everything drops to 0.

In [ ]:
W = 8.0
show(book.sweep('3_sweep_offset', h.offset_case, [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 3.9, 4.2],
                ['Q_pt', 'Q_od', 'Q_v(c)', 'D_v'], xlabel='parallel offset (pt)',
                refs={'exp(-2x/W)': lambda x: math.exp(-2 * x / W) if x < W / 2 else 0}))
show(book.sweep('3_sweep_skew', h.skew_case, [0, 0.5, 1, 1.5, 2, 2.5, 3, 3.5, 3.9],
                ['Q_pt', 'Q_od', 'Q_v(c)', 'D_v'], xlabel='one end offset (pt)',
                refs={'exp(-x/W)': lambda x: math.exp(-x / W), 'exp(-2x/W)': lambda x: math.exp(-2 * x / W)}))
show(book.sweep('3_sweep_width', h.width_case, [0.25, 0.5, 0.75, 1, 1.25, 1.5, 2, 2.5],
                ['Q_w', 'Q_v(c)', 'D_v'], xlabel='W(k) / W(g)',
                refs={'exp(-|r-1|)': lambda r: math.exp(-abs(r - 1))}))

## 4 - Per-line quality: Q_b, Q_fr, Q_v (§4)

Per GT line g, over D(g) = the predictions overlapping it:

- **Q_b(g)** = sum Q_v(c) l(c) / max(l(g), sum l(c)) -- *coverage x overlap quality*; uncovered length counts 0.
- **Q_fr(g)** = sqrt(sum l(c)^2) / sum l(c) -- *fragmentation*: 1 for one piece, 1/sqrt(n) for n equal pieces.
- **Q_v(g)** = Q_b x Q_fr; its length-weighted mean over GT is **D_v**.

The same on the prediction side over G(k) = the GT lines k overlaps: Q_fr(k) measures
**consolidation** (one prediction across n GT lines), and F_v(k) = 1 - Q_v(k); its length-weighted
mean is **F_v**. Watch the colours: a fragmented GT line turns yellow on the GT band while its pieces
stay green; a consolidated prediction is yellow while the GT items stay green.

In [ ]:
lines = h.run_all(h.line_cases())
show(book.cases('4_lines', lines, ['Q_b(g)', 'Q_fr(g)', 'Q_v(g)', 'Q_b(k)', 'Q_fr(k)', 'Q_v(k)', 'D_v', 'F_v']))
show(book.sweep('4_sweep_coverage', h.coverage_case, [0.1, 0.25, 0.4, 0.5, 0.6, 0.75, 0.9, 1.0],
                ['Q_b(g)', 'D_v', 'F_v'], xlabel='covered fraction', refs={'x': lambda x: x}))
show(book.sweep('4_sweep_fragments', h.fragment_case, [1, 2, 3, 4, 5, 6, 8],
                ['Q_fr(g)', 'Q_b(g)', 'D_v', 'F_v'], xlabel='equal pieces n',
                refs={'1/sqrt(n)': lambda n: 1 / math.sqrt(n)}))
show(book.sweep('4_sweep_consolidation', h.consolidation_case, [1, 2, 3, 4, 5, 6, 8],
                ['Q_fr(k)', 'D_v', 'F_v'], xlabel='GT items under one prediction n',
                refs={'1/sqrt(n)': lambda n: 1 / math.sqrt(n), '1 - 1/sqrt(n)': lambda n: 1 - 1 / math.sqrt(n)}))

Guide values: Q_fr 0.71 ~ 2 pieces, 0.58 ~ 3, 0.5 ~ 4.

## 5 - Page level: D_v, F_v, VRI (§4, §10)

- **D_v** = sum Q_v(g) l(g) / sum l(g): *quality-weighted recall* -- how much GT length was found, and how well.
- **F_v** = sum (1 - Q_v(k)) l(k) / sum l(k): *quality-weighted 1 - precision*.
- **VRI** = 0.5 D_v + 0.5 (1 - F_v).

Things to notice:

- Everything is weighted by **length**, not line count.
- **Exact duplicates are invisible to F_v** (each copy scores Q_v(k) = 1); they only show as GT
  fragmentation (|D(g)| = 2, D_v 0.71) and length ratio 2.
- Rows A and B have the **same VRI = 0.75** for opposite reasons (finds everything with as much junk
  vs. finds half, no junk) -- always read VRI with D_v and F_v.
- Predicting nothing gives D_v 0 and F_v / VRI **n/a**, not VRI 0.5.

In [ ]:
page = h.run_all(h.page_cases())
show(book.cases('5_page', page, ['D_v', 'F_v', 'VRI', 'Q_fr(g)', 'Q_fr(k)', '|D(g)|', '|G(k)|', 'length ratio']))

## 6 - Fragmentation / consolidation histograms (§4)

Per GT line, **|D(g)|** = how many predicted bars overlap it (0 = missed, 1 = clean, 2+ = fragmented);
per prediction, **|G(k)|** = how many GT bars it overlaps (0 = pure false alarm, 2+ = consolidated).
They count **lines, not length**; "mean (matched)" excludes the 0 bucket.

The drawing below has one of each: the top edge is stored as 2 CAD items under one prediction
(|G(k)| 2), the right edge is traced in 3 pieces (|D(g)| 3), the bottom edge and one hatch line are
missed (|D(g)| 0), hatch 70 is in 2 pieces, the diagonal is traced twice (|D(g)| 2) and a noise stroke
overlaps nothing (|G(k)| 0).

In [ ]:
drawing = h.run(h.drawing_case())
show(book.histograms('6_histograms', drawing))
table([drawing], ['D_v', 'F_v', 'VRI', 'Q_b(g)', 'Q_fr(g)', 'Q_fr(k)', 'mean |D(g)|', 'mean |G(k)|'])

## 7 - Splits (§5)

- **text / non-text** -- a bar is "text" when > 50 % of it lies inside one text label box (both
  sides). RasterText erases text before tracing, so a *successful* erase gives text D_v 0 while the
  non-text (tracing) score stays clean; unerased, badly traced text raises text F_v.
- **solid / dashed** (GT only, F_v n/a) -- a dashed GT line is *one* line; P2 traces its dashes as
  solid pieces, which leaves the gaps uncovered (Q_b <= dash / period = 6/9) and fragments it
  (Q_fr ~ 1/sqrt 10). Dashed D_v is low by construction; compare it between runs only.
- **GT type** (GT only) -- `vector_to_raster` vs hand-traced `original_raster`.

In [ ]:
splits = h.run_all(h.split_cases())
show(book.cases('7_splits', splits, ['D_v', 'F_v', 'D_v text', 'F_v text', 'D_v non-text', 'F_v non-text',
                                    'D_v solid', 'D_v dashed', 'D_v vector_to_raster', 'D_v original_raster']))

## 8 - Pixel level: D_p, F_p, PRI vs D_v, F_v (§6)

GT and predicted bars are rendered black at W_eff (300 dpi), ink = gray < 128:
**D_p** = |GT ink & pred ink| / |GT ink|, **F_p** = 1 - |both| / |pred ink|, **PRI** = 0.5 D_p + 0.5 (1 - F_p).
The pixel metrics only ask "is the ink in the right place" -- they are blind to fragmentation,
endpoints and line identity. Comparing the two pairs is the main diagnostic:

- fragmented (tiny gaps): D_p ~ 1 while D_v ~ 1/sqrt 8 -> *the ink is right, the line structure isn't*;
- a pixel staircase along a 26.6 deg line: every horizontal / vertical step fails the 15 deg gate ->
  D_v 0 and F_v 1, though D_p is ~0.9;
- noise strokes on paper raise **both** F_p and F_v.

Middle columns: the ink overlay (D_p / F_p input) and the grayscale renders SSIM / PSNR compare.

In [ ]:
pixel = h.run_all(h.pixel_cases())
show(book.pixels('8_pixel', pixel, ['D_v', 'F_v', 'VRI', 'D_p', 'F_p', 'PRI', 'Q_fr(g)', '|G(k)|']), width=1200)

## 9 - SSIM / PSNR (§6)

Whole-page grayscale renders of the raw geometry at real widths (150 dpi): SSIM with a Gaussian window,
PSNR capped at 100 dB. Both are dominated by **blank paper**: the sparse 400 x 300 page misses a third
of its lines and still scores SSIM ~ 0.99, the dense page missing one line in nine scores lower, and a
uniform 1 pt shift drives SSIM to ~0 while the line count is perfect. Sanity checks only -- never rank
backends on them.

In [ ]:
ssim = h.run_all(h.ssim_cases())
show(book.pixels('9_ssim', ssim, ['D_v', 'F_v', 'D_p', 'F_p', 'SSIM', 'PSNR']), width=1200)

## 10 - Global sanity checks: length ratio and direction (§7)

- **length ratio** = sum predicted length / sum GT length (raw flattened geometry, unbounded): > 1 =
  over-tracing / duplicates (a thick stroke traced on both edges = 2), < 1 = misses.
- **direction intersection** = overlap of the two length-weighted 36-bin (5 deg) angle histograms
  (mod 180). It catches whole-page coordinate problems: transposed coordinates keep the length ratio
  at 1 but swap the 0 / 90 deg peaks; a 10 deg rotation moves every line out of its bin. In both cases
  every per-line metric just reads "missed" -- this is the number that tells you *why*.

In [ ]:
sanity = h.run_all(h.sanity_cases())
show(book.direction('10_direction', sanity))
table(sanity, ['D_v', 'F_v', 'D_p', 'length ratio', 'direction'])

## 11 - Known biases (§11)

Accepted consequences of the protocol, so a real page can't reach 1:

- **duplicate GT** (CAD drew a line twice): one perfect prediction overlaps two GT lines -> Q_fr(k) 0.71, F_v 0.29;
- **CAD-split GT** (one physical line stored as 3 items): a clean single prediction looks consolidated (|G(k)| 3);
- **curves**: GT and prediction are RDP'd independently, so even an exact trace split into two cubic
  halves breaks into bars at different points -> some |D(g)| / |G(k)| = 2 and a lower direction score
  (the 5 deg bins are sensitive to chord angles);
- **abutting items**: a prediction poking 2 pt into its neighbour's GT item is accepted by the
  endpoint rule, adding a short overlap and one to |D(g)|.

In [ ]:
bias = h.run_all(h.bias_cases())
show(book.cases('11_biases', bias, ['D_v', 'F_v', 'Q_fr(g)', 'Q_fr(k)', '|D(g)|', '|G(k)|', 'direction']))

## 12 - Diagnostic cheat-sheet (§10)

One case per row of the doc's "Diagnostic patterns" table, with the numbers that identify it.
Read across a row: the *combination* of D_v / F_v with D_p / F_p, Q_fr, length ratio and direction
names the cause.

In [ ]:
diag = h.run_all(h.diagnostic_cases())
table(diag, ['D_v', 'F_v', 'D_p', 'F_p', 'Q_pt', 'Q_od', 'Q_fr(g)', 'Q_fr(k)', '|G(k)|', 'length ratio', 'direction'])
print(len(book.saved), 'images in', book.root)